<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a>, <a href="https://www.hse.ru/en/staff/sara/" target="_blank">Saraa Ali</a>  ©2026 onwards</font></small><hr style="margin:0;background-color:silver">

**[<font size=6>🚗Auto</font>](https://www.kaggle.com/t/f9ccca19d93b444fa7f402b7d86a53ad)**. [**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

<small>**(Optional) CONSENT.** <mark>[ ]</mark> We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is optional and this decision will not affect our grade in any way. <font color=gray><i>(If ok with sharing your Colab for educational purposes, leave "X" in the check box.)</i></font></small>

In [9]:
%%time
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import pandas as pd, time, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print('timer started')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=10000, precision=4, edgeitems=20, suppress=True)
pd.set_option('display.max_rows', 100, 'display.max_columns', 100, 'display.max_colwidth', 100, 'display.precision', 2, 'display.max_rows', 4)

db = fetch_openml('BNG(auto_price)')   # load databunch (dictionary)
tX = pd.DataFrame(db['data'], columns=db['feature_names'])
tX.symboling = tX.symboling.astype('float')
tX['price'] = db['target']
YCols = ['city-mpg','highway-mpg','price']  # 3 targets
tY = tX[YCols]
tX.drop(YCols, axis=1, inplace=True)
# tY = pd.Series(db['target'], name='price')
tX, vX, tY, DO_NOT_USE = train_test_split(tX, tY, train_size=0.7, random_state=0, shuffle=True)
# ToCSV(DO_NOT_USE, 'testY')   # Students cannot use these test values
del DO_NOT_USE
tX
tY
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

timer started
CPU times: user 1.78 s, sys: 324 ms, total: 2.11 s
Wall time: 2.21 s


In [10]:
tmr = Timer()

timer started


<hr color=red>

<font size=5>⏳</font> <strong><font color=orange size=5>Your Code, Documentation, Ideas and Timer - All Start Here...</font></strong>

**Student's Section** (between ⏳ symbols): add your code and documentation here.

In [ ]:
%%time
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from scipy.optimize import minimize

good_rows = tY.notna().all(axis=1) & (tY['price'] > 0) & (tY['city-mpg'] > 0) & (tY['highway-mpg'] > 0)
X_train_full = tX.loc[good_rows].copy()
y_train_full = tY.loc[good_rows].copy()

y_train_log = y_train_full.copy()
y_train_log['price'] = np.log1p(y_train_full['price'])

rng = np.random.RandomState(0)
shuffled = rng.permutation(len(X_train_full))
val_size = 25_000
val_ids = shuffled[:val_size]
train_ids = shuffled[val_size:]

X_train = X_train_full.iloc[train_ids]
y_train = y_train_log.iloc[train_ids]
X_val = X_train_full.iloc[val_ids]
y_val = y_train_log.iloc[val_ids]

model_ridge = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('spline', SplineTransformer(n_knots=5, degree=3, include_bias=False)),
    ('ridge',  RidgeCV(alphas=np.logspace(-2, 2, 6))),
])
model_ridge.fit(X_train, y_train)

sample_size = 200_000
sample_ids = rng.choice(len(X_train), sample_size, replace=False)
model_knn = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('spline', SplineTransformer(n_knots=4, degree=3, include_bias=False)),
    ('pca',    PCA(n_components=20, random_state=0)),
    ('knn',    KNeighborsRegressor(n_neighbors=30, weights='distance', n_jobs=-1)),
])
model_knn.fit(X_train.iloc[sample_ids], y_train.iloc[sample_ids])

model_kernel = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('rbf',    Nystroem(kernel='rbf', gamma=0.05, n_components=400, random_state=0, n_jobs=-1)),
    ('ridge',  Ridge(alpha=1.0, random_state=0)),
])
model_kernel.fit(X_train, y_train)


preds_ridge_val  = model_ridge.predict(X_val)
preds_knn_val    = model_knn.predict(X_val)
preds_kernel_val = model_kernel.predict(X_val)


def to_orig(predictions):
    result = predictions.copy()
    result[:, 2] = np.expm1(result[:, 2])  
    return result

y_val_original = to_orig(y_val.values)

def flat_r2(predictions_log):
    return r2_score(y_val_original.flatten(), to_orig(predictions_log).flatten())


def loss_function(weights_flat):
    w = np.abs(weights_flat).reshape(3, 3)
    w = w / w.sum(axis=0, keepdims=True)  
    blended = np.zeros_like(preds_ridge_val)
    for target_idx in range(3):
        blended[:, target_idx] = (w[0, target_idx] * preds_ridge_val[:, target_idx]
                                + w[1, target_idx] * preds_knn_val[:, target_idx]
                                + w[2, target_idx] * preds_kernel_val[:, target_idx])
    return -flat_r2(blended)

start_weights = np.ones(9) / 3
optimization_result = minimize(loss_function, start_weights, method='Nelder-Mead',
                               options={'maxiter': 500, 'xatol': 1e-4})

best_weights = np.abs(optimization_result.x).reshape(3, 3)
best_weights = best_weights / best_weights.sum(axis=0, keepdims=True)

print(f'Validation FLAT R2: Ridge={flat_r2(preds_ridge_val):.4f} | '
      f'KNN={flat_r2(preds_knn_val):.4f} | '
      f'Kernel={flat_r2(preds_kernel_val):.4f}')
print(f'Validation FLAT R2 (Blend): {-optimization_result.fun:.4f}')


preds_ridge_test  = model_ridge.predict(vX)
preds_knn_test    = model_knn.predict(vX)
preds_kernel_test = model_kernel.predict(vX)

final_predictions = np.zeros_like(preds_ridge_test)
for target_idx in range(3):
    final_predictions[:, target_idx] = (best_weights[0, target_idx] * preds_ridge_test[:, target_idx]
                                      + best_weights[1, target_idx] * preds_knn_test[:, target_idx]
                                      + best_weights[2, target_idx] * preds_kernel_test[:, target_idx])

submission = pd.DataFrame(final_predictions, index=vX.index, columns=YCols)
submission['price'] = np.expm1(submission['price']) 


for col in YCols:
    low, high = y_train_full[col].quantile([0.001, 0.999])
    submission[col] = submission[col].clip(lower=low, upper=high)

submission = submission.replace([np.inf, -np.inf], np.nan).fillna(y_train_full.median())

ToCSV(submission, 'submission')
tmr.ShowTime()


Validation FLAT R2: Ridge=0.8536 | KNN=0.8685 | Kernel=0.8614
Validation FLAT R2 (Blend): 0.8690
Runtime is 50 sec
CPU times: user 3min 55s, sys: 11.7 s, total: 4min 6s
Wall time: 48.6 s


In [5]:
sub = pd.read_csv('submission.csv')
sub.to_csv('submission.csv.gz', index=False, compression='gzip')

## **Task 1. Preprocessing Pipeline**

Explain elements of your preprocessing pipeline i.e. feature engineering, subsampling, clustering, dimensionality reduction, etc.
1. Why did you choose these elements? (Something in EDA, prior experience,...? Btw, EDA is not required)
1. How do you evaluate the effectiveness of these elements?
1. What else have you tried that worked or didn't?

**Student's answer:**

**Student's answer:**

**1. Cleaning targets.** To start, I filter out rows with NaN or price/mpg <= 0. I only managed to remove a few rows (a few per 700k), but this is definitely necessary for stability.

**2. Logarithm of price.** When predicting price directly, expensive machines add error, and linear models perform poorly. I applied np.log1p to price before training, and then used np.expm1 for the final predictions. This improved R² specifically on the price target..

**3. Median-imputer and StandardScaler.** I fill gaps with the median (robust against outliers), and standardize all numerical features to zero mean and unit variance. This is necessary for KNN, PCA, Nystroem, and Ridge with regularization, as these models are sensitive to scale.

**4. SplineTransformer.** Linear Ridge yielded an R² of only 0.32 in the baseline, indicating that the relationships between features and targets are nonlinear. Splines expand each numerical feature into several new ones, allowing the linear model to capture nonlinearities without manually constructing interactions. I tried PolynomialFeatures(degree=2), but splines performed better with lower dimensions.

**5. PCA (20) before KNN.** KNN searches for nearest neighbors based on distance, and in a space with a lot of features, distances become indistinguishable. PCA compresses the data to 20 directions, where the data is most scattered, so KNN finds truly similar cars and runs faster. The number of components (20) was chosen based on the holdout rate: below 15, R² is falling; above 25, KNN slows down without improving quality.

**6. Nystroem (RBF-ядро).** This method approximates the features of an RBF kernel without constructing a full 700k × 700k kernel matrix. The idea is to select a small number of control points, calculate the distances from each object to them, and use these distances as new features. The resulting model behaves similarly to an SVM with an RBF kernel. The standard Ridge algorithm operates on the basis of these features.

**7. 200k subsample for KNN.** On the full 700k dataset, KNN doesn't fit within the time limit. I experimentally found that 200k gives almost the same performance as the full dataset, but faster.

**8. Clip predictions for quantiles.** Just a protection from random extreme predictions.

**How I evaluated effectiveness.** I had a separate holdout (25k rows, which was not used for training), and I calculated flat R² on it. This allowed me to quickly test ideas without submitting. I also compared individual models and blends. If a single model is almost equal to the blend, ensemble is pointless.

**What didn't work:**
 - **PolynomialFeatures(degree=2)** - gave about the same score as splines, but created twice as many features and took longer to train.
- **Increasing N_SAMPLE for KNN to 250k and k to 40** - gave +1% to the score, but exceeded the 60-second limit.
- **Clipping predictions by min/max train instead of 0.1%–99.9% quantiles** - left too extreme values, and the final score was slightly worse.

## **Task 2. Modeling Approach**
Explain your modeling approach, i.e. ideas you tried and why you thought they would be helpful.

1. How did these decisions guide you in modeling?
1. How do you evaluate the effectiveness of these elements?
1. What else have you tried that worked or didn't?

**Student's answer:**

My approach is an ensemble of 3 models with weights optimized for R². The idea is that each model captures a different type of dependency in the data, and blending them averages out individual errors.

**Model 1: Ridge + Splines.** A linear model with L2 regularization on top of B-spline features (degree 3). Splines transform numeric features into nonlinear basis functions, allowing Ridge to capture nonlinear relationships while staying linear in parameters. I used RidgeCV to auto-tune alpha.

**Model 2: KNN + PCA + Splines.** Predicts based on the 30 nearest neighbors with distance-weighted averaging. PCA reduces dimensionality to 20 components before KNN, otherwise distances become meaningless in high-dimensional space. Trained on a 200k subsample. This model is the strongest on the price target.

**Model 3: Nystroem(RBF) + Ridge.** Model 3: Nystroem(RBF) + Ridge. Nystroem generates 400 nonlinear features based on RBF distances to random anchor points, and Ridge runs on top. This is a scalable approximation of SVM with an RBF kernel, since a full SVM doesn't train on 700k rows in reasonable time.

**Per-target ensemble blending.** I use 9 weights (3 models × 3 targets), optimized via scipy.minimize directly on R² evaluated on the hold-out set. This lets each model contribute optimally to each target separately (for example, the optimal weights for price differ significantly from those for city-mpg)

**How I evaluated effectiveness.** I held out 25k rows that were never used for training. On this set I computed R² for each individual model and for the blend. Final blend weights were tuned on this same hold-out.

**What didn't work:**
- **RegressorChain (target stacking).** The idea was to predict highway-mpg first and use it as a feature for city-mpg. In practice, the score dropped.
- **Refitting models on the full dataset before final prediction.** Should give a small boost in theory, but added 30 seconds and broke the time limit.
- **Pure KNN without blending on price** the score was a little bit lower.

Below is a baseline model that produces the result on Kaggle leaderboard (LB).

In [ ]:
m = MultiOutputRegressor(Ridge(random_state=0)).fit(tX, tY)  # always seed RNG for reproducibility
print(f'In-sample R^2 = {m.score(tX,tY):.4f}')

In-sample R^2 = 0.3245


In [ ]:
pY = pd.DataFrame(m.predict(vX), index=vX.index, columns=YCols)
ToCSV(pY, 'Auto_baseline')

# **References:**

1. Remember to cite your sources here as well! At the least, your textbook should be cited. Google Scholar allows you to effortlessly copy/paste an APA citation format for books and publications. Also cite StackOverflow, package documentation, and other meaningful internet resources to help your peers learn from these (and to avoid plagiarism claims).

Ridge regression docs https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html
KNN https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html
Kernel Approximation (Nystroem method) https://scikit-learn.org/stable/modules/generated/sklearn.kernel_approximation.Nystroem.html
Spline Transformer https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.SplineTransformer.html
PCA https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
https://scikit-learn.org/stable/modules/decomposition.html

An Introduction to Statistical Learning textbook

<font color=red><b>Your answer here.</b></font>

<font size=5>⌛</font> <strong><font color=orange size=5>Do not exceed competition's runtime limit!</font></strong>

<hr color=red>


In [12]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 57 sec


## 💡**Starter Ideas**

1. Tune model hyperparameters and try different allowed models
1. Try to linear and non-linear feature normalization: shift/scale, log, divide features by features (investigate scatterplot matrix)
1. Try higher order feature interactions and polynomial features on a small subsample. Then identify key features or select key principal components. The final model can be trained on a larger or even full training sample. You can use [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) to reduce the feature set
1. Do a thorough EDA: look for feature augmentations that result in linear decision boundaries between pairs of classes.
1. Evaluate predictions and focus on poorly predicted "groups":
  1. Strongest errors. E.g. the model is very confident about the wrong label
1. Do scatter plots show piecewise linear shape? Can a separate linear model be used on each support, or can the pattern be linearized via transformations?
1. Try modeling each output separately from inputs or from a other modeled output
1. Try stepwise selection and regularization and remove "unimportant" features from final model